## Key findings — HAM10000 metadata

*All figures at lesion level (7,470 lesions) unless stated. One mole photographed
six times is not six examples.*

**1. Split on `lesion_id`, not on rows.** 7,470 lesions produced 10,015 images;
1,956 lesions have 2–6 photos each, accounting for 45% of the images. A random
row split puts the same mole in train and test, and our scores become memory
rather than skill. Must be settled before anyone trains.

**2. One source is over half the dataset and almost disease-free.**
`vidir_molemax` is **52.9%** of our lesions (3,954) and **0.4%** referable —
about 16 lesions. Rosendahl, by contrast, is 41.7%. MoleMax also carries *all*
3,704 `follow_up` labels, while rosendahl and vienna_dias are 100% histo. So
source, verification method and diagnosis are one correlation, not three: a
monitoring device for known-benign moles generated half our data.
Counting images understates this (39.5%), because the *other* sources are the
ones with repeat photos — vidir_modern 1.98 images per lesion, vienna_dias
1.58, rosendahl 1.46, MoleMax exactly 1.00.
*Risk:* cameras leave fingerprints — resolution, colour, vignetting — so the
model can score well by recognising the device rather than reading the lesion.
*Test:* hold out `rosendahl` entirely (1,552 lesions, 100% histo, different
continent and camera). If accuracy survives, we learned dermatology.

**3. Our positive class is smaller than it looks.** 1,169 referable lesions
against 6,301 benign — 15.6%, not the 19.5% the image counts suggest. Melanoma
is **614 distinct lesions**, not 1,113 images. That's a modest number of unique
positives for a deep network, and it should shape how much we expect from
training from scratch versus transfer learning.

**4. Labels: refer = mel, bcc, akiec.** Melanoma plus the other skin cancers —
an app that waves through a basal cell carcinoma has failed even though it
isn't melanoma. Note `bkl` (727 lesions) sits in "monitor" but is the classic
melanoma mimic; expect it to dominate our false positives, and that is correct
behaviour for a triage tool.

**5. Two things that aren't the lesion predict the label.** Referable lesions
are 14 years older on average (63.3 vs 49.1), and referral rate varies 10× by
body site (face 42.7%, trunk 4.2%) because sun-exposed skin carries the
keratoses and carcinomas. The model can use skin ageing or visible face/ear
context instead of lesion morphology — and our users photograph backs and
arms, where it would have no such advantage.
*Test:* Grad-CAM on a sample of correct predictions shows us where it looked.

**6. Option: evaluate on histo-confirmed images only.** The histo subset holds
*all* referable cases (akiec, bcc and mel are 100% histo), so restricting to it
costs zero positives, balance improves, and MoleMax nearly disappears.
*But* histo-confirmed **benign** lesions are the ones a dermatologist thought
worth cutting — training on those alone deletes every ordinary mole, which is
exactly what our users will send.
*Proposal:* train on everything, evaluate on histo only.

**7. Every image is dermatoscopic. None are phone photos.** All 10,015 were
taken with a contact dermatoscope: lesion filling the frame, flat polarised
light, no surrounding skin. A phone photo is a different kind of image, not a
harder version of the same one — this is what the BMJ review blames for
commercial skin apps underperforming. Not a blocker for week one, but the app
should state what it was built on.

---

## Proposed plan

- **Split** on `lesion_id` — before anyone trains
- **Train** on all 10,015 images — keeps the ordinary moles the model needs
- **Report two numbers, not one:**
  - *headline* — lesion-level random split, evaluated on histo-confirmed images
  - *honesty check* — `rosendahl` held out entirely, never seen in training

  The gap between them is itself a result worth presenting.
- **If week two allows:** run the trained model over PAD-UFES-20 (2,298
  smartphone images, 52 melanomas) as a test set — no retraining needed — and
  Grad-CAM on a sample of correct predictions.

| code | diagnosis | |
|---|---|---|
| nv | melanocytic nevus | benign |
| mel | melanoma | malignant |
| bkl | benign keratosis | benign |
| bcc | basal cell carcinoma | malignant |
| akiec | actinic keratosis / Bowen's | pre-cancerous |
| vasc | vascular lesion | benign |
| df | dermatofibroma | benign |

In [1]:
import pandas as pd
df = pd.read_csv("../raw_data/ham10000/HAM10000_metadata")
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern


In [2]:
df.shape

(10015, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  object 
 1   image_id      10015 non-null  object 
 2   dx            10015 non-null  object 
 3   dx_type       10015 non-null  object 
 4   age           9958 non-null   float64
 5   sex           10015 non-null  object 
 6   localization  10015 non-null  object 
 7   dataset       10015 non-null  object 
dtypes: float64(1), object(7)
memory usage: 626.1+ KB


In [5]:
df.describe(include='all')

,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
count,10015,10015,10015,10015,9958.000000,10015,10015,10015
unique,7470,10015,7,4,NaN,3,15,4
top,HAM_0003789,ISIC_0027419,nv,histo,NaN,male,back,vidir_molemax
freq,6,1,6705,5340,NaN,5406,2192,3954
mean,NaN,NaN,NaN,NaN,51.863828,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,16.968614,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,40.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,50.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,65.000000,NaN,NaN,NaN


In [6]:
df.nunique()

lesion_id        7470
image_id        10015
dx                  7
dx_type             4
age                18
sex                 3
localization       15
dataset             4
dtype: int64

In [7]:
for col in ["dx", "dx_type", "localization", "dataset"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- dx ---
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

--- dx_type ---
dx_type
histo        5340
follow_up    3704
consensus     902
confocal       69
Name: count, dtype: int64

--- localization ---
localization
back               2192
lower extremity    2077
trunk              1404
upper extremity    1118
abdomen            1022
face                745
chest               407
foot                319
unknown             234
neck                168
scalp               128
hand                 90
ear                  56
genital              48
acral                 7
Name: count, dtype: int64

--- dataset ---
dataset
vidir_molemax    3954
vidir_modern     3363
rosendahl        2259
vienna_dias       439
Name: count, dtype: int64


In [9]:
pd.crosstab(df.dx, df.dx_type)

dx_type,confocal,consensus,follow_up,histo
dx,,,,
akiec,0,0,0,327
bcc,0,0,0,514
bkl,69,264,0,766
df,0,60,0,55
mel,0,0,0,1113
nv,0,503,3704,2498
vasc,0,75,0,67


In [10]:
pd.crosstab(df.dataset, df.dx, normalize="index").round(3)

dx,akiec,bcc,bkl,df,mel,nv,vasc
dataset,,,,,,,
rosendahl,0.131,0.131,0.217,0.013,0.151,0.355,0.001
vidir_modern,0.010,0.063,0.141,0.015,0.202,0.545,0.024
vidir_molemax,0.000,0.001,0.031,0.008,0.006,0.941,0.014
vienna_dias,0.000,0.011,0.023,0.009,0.153,0.797,0.007


In [11]:
pd.crosstab(df.dataset, df.dx)

dx,akiec,bcc,bkl,df,mel,nv,vasc
dataset,,,,,,,
rosendahl,295,296,490,30,342,803,3
vidir_modern,32,211,475,51,680,1832,82
vidir_molemax,0,2,124,30,24,3720,54
vienna_dias,0,5,10,4,67,350,3


In [12]:
print((df['age'] == 0).sum())

39


In [13]:
df[df.age == 0].dx.value_counts()

dx
nv      30
bkl      5
vasc     4
Name: count, dtype: int64

In [14]:
REFER = {
    "mel":   1,    # melanoma
    "bcc":   1,    # basal cell carcinoma — a skin cancer
    "akiec": 1,    # actinic keratosis / Bowen's — pre-cancerous
    "nv":    0,    # ordinary mole
    "bkl":   0,    # benign keratosis
    "vasc":  0,    # vascular lesion
    "df":    0,    # dermatofibroma
}

df["refer"] = df.dx.map(REFER)
df.refer.value_counts()

refer
0    8061
1    1954
Name: count, dtype: int64

In [15]:
pd.crosstab(df.refer, df.dataset, normalize="columns").round(3)

dataset,rosendahl,vidir_modern,vidir_molemax,vienna_dias
refer,,,,
0,0.587,0.726,0.993,0.836
1,0.413,0.274,0.007,0.164


In [16]:
df.groupby("refer").age.describe()

,count,mean,std,min,25%,50%,75%,max
refer,,,,,,,,
0,8006.0,49.080689,16.344335,0.0,40.0,50.0,60.0,85.0
1,1952.0,63.278689,14.534753,5.0,55.0,65.0,75.0,85.0


In [17]:
pd.crosstab(df.localization, df.refer, normalize="index").round(3).sort_values(1, ascending=False)

refer,0,1
localization,,
face,0.573,0.427
scalp,0.633,0.367
ear,0.643,0.357
neck,0.685,0.315
chest,0.688,0.312
upper extremity,0.710,0.290
back,0.754,0.246
hand,0.822,0.178
lower extremity,0.848,0.152


In [24]:
pd.crosstab(df.dataset, df.dx_type)

dx_type,confocal,consensus,follow_up,histo
dataset,,,,
rosendahl,0,0,0,2259
vidir_modern,69,700,0,2594
vidir_molemax,0,202,3704,48
vienna_dias,0,0,0,439


In [25]:
histo = df[df.dx_type == "histo"]
print(len(histo))
print(histo.refer.value_counts())
print(histo.dataset.value_counts())

5340
refer
0    3386
1    1954
Name: count, dtype: int64
dataset
vidir_modern     2594
rosendahl        2259
vienna_dias       439
vidir_molemax      48
Name: count, dtype: int64


In [26]:
df.groupby("dataset").agg(
    images=("image_id", "count"),
    lesions=("lesion_id", "nunique"),
).assign(per_lesion=lambda x: (x.images / x.lesions).round(2))

,images,lesions,per_lesion
dataset,,,
rosendahl,2259,1552,1.46
vidir_modern,3363,1695,1.98
vidir_molemax,3954,3954,1.00
vienna_dias,439,278,1.58


In [28]:
lesions = df.drop_duplicates("lesion_id")
print(len(lesions))
print(lesions.refer.value_counts())
print(pd.crosstab(lesions.dataset, lesions.refer, normalize="index").round(3))
pd.crosstab(lesions.dx, lesions.dx_type)

7470
refer
0    6301
1    1169
Name: count, dtype: int64
refer              0      1
dataset                    
rosendahl      0.583  0.417
vidir_modern   0.730  0.270
vidir_molemax  0.996  0.004
vienna_dias    0.827  0.173


dx_type,confocal,consensus,follow_up,histo
dx,,,,
akiec,0,0,0,228
bcc,0,0,0,327
bkl,34,206,0,487
df,0,44,0,29
mel,0,0,0,614
nv,0,332,3704,1367
vasc,0,65,0,33


In [29]:
lesions.dx.value_counts()

dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64